load data

In [1]:
import pandas as pd
import pm4py

log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")
df = pm4py.convert_to_dataframe(log)

/Users/felixhauptmann/ProM-Assignment-Group-C/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

## Unanchored Events
Problem:
- different tools and logs make use of different formats for time data
- likely to occur if event log is constructed from multiple sources

Manifestation & Detection:
- timestamp data from one original source differs significantly from data coming from another original source
- missing timestamp data across many events in log
- values outside of the expected range, for instance "1st in the 14th monts of 2026"

In [2]:
print(df["EventOrigin"].unique())
print(df.columns.tolist())

<StringArray>
['Application', 'Workflow', 'Offer']
Length: 3, dtype: str
['Action', 'org:resource', 'concept:name', 'EventOrigin', 'EventID', 'lifecycle:transition', 'time:timestamp', 'case:LoanGoal', 'case:ApplicationType', 'case:concept:name', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount', 'OfferID']


In [3]:
# simulate different data sources (for unique values of EventOrigin)
mask_app = df["EventOrigin"] == "Application"
mask_offer = df["EventOrigin"] == "Offer"

# convert timestamp objects into strings
df["time:timestamp"] = df["time:timestamp"].astype("string")

# change the format of timestamp for all rows of a simulated data source
    # format: dd.mm.YYYY HH:MM:SS
df.loc[mask_app, "time:timestamp"] = (
    pd.to_datetime(df.loc[mask_app, "time:timestamp"], errors="coerce")
    .dt.strftime("%d.%m.%Y %H:%M:%S")
)

# format: YYYY-mm-dd'T'HH:MM:SS:MS'Z'
df.loc[mask_offer, "time:timestamp"] = (
    pd.to_datetime(df.loc[mask_offer, "time:timestamp"], errors="coerce")
    .dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)


In [4]:
# output to show heterogenous timestamp formats of different simulated data sources (EventOrigin) 
for origin in ["Application", "Workflow", "Offer"]:
    print("\n", origin)
    print(
        df[df["EventOrigin"] == origin]
        .sort_values("time:timestamp")
        .head(5)[["EventOrigin", "time:timestamp"]]
    )


 Application
    EventOrigin       time:timestamp
0   Application  01.01.2016 09:51:15
1   Application  01.01.2016 09:51:15
5   Application  01.01.2016 09:52:36
41  Application  01.01.2016 10:16:11
40  Application  01.01.2016 10:16:11

 Workflow
   EventOrigin                    time:timestamp
2     Workflow  2016-01-01 09:51:15.774000+00:00
3     Workflow  2016-01-01 09:52:36.392000+00:00
4     Workflow  2016-01-01 09:52:36.403000+00:00
42    Workflow  2016-01-01 10:16:11.740000+00:00
43    Workflow  2016-01-01 10:17:31.573000+00:00

 Offer
    EventOrigin               time:timestamp
917       Offer  2016-01-02T09:17:05.720000Z
918       Offer  2016-01-02T09:17:08.762000Z
919       Offer  2016-01-02T09:19:21.330000Z
924       Offer  2016-01-02T09:21:26.034000Z
925       Offer  2016-01-02T09:21:42.022000Z


In [5]:
df.head(10)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,01.01.2016 09:52:36,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Obtained,User_17,W_Complete application,Workflow,Workitem_1875340971,start,2016-01-02 10:45:22.429000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Released,User_17,W_Complete application,Workflow,Workitem_1452291795,suspend,2016-01-02 10:49:28.816000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,02.01.2016 11:23:04,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Created,User_52,O_Create Offer,Offer,Offer_148581083,complete,2016-01-02T11:29:03.994000Z,Existing loan takeover,New credit,Application_652823628,20000.0,20000.0,44.0,True,498.29,True,979.0,20000.0,NaN


In [18]:
# converting it back to xes
df_export = df.copy()

# df_export["time:timestamp"] = pd.to_datetime(
#     df_export["time:timestamp"],
#     errors="coerce",
#     utc=True,
# )

# event_log = pm4py.convert_to_event_log(df_export)
# pm4py.write_xes(event_log, "noised.xes")

df_export.head(10)

#df_export.to_csv("../data/event_log_cleaned.csv", index=False, sep=";", encoding="utf-8")
df_export.to_csv("noised.csv", index=False, sep=";")

In [17]:
df_export.head(10)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,01.01.2016 09:52:36,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Obtained,User_17,W_Complete application,Workflow,Workitem_1875340971,start,2016-01-02 10:45:22.429000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Released,User_17,W_Complete application,Workflow,Workitem_1452291795,suspend,2016-01-02 10:49:28.816000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,02.01.2016 11:23:04,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Created,User_52,O_Create Offer,Offer,Offer_148581083,complete,2016-01-02T11:29:03.994000Z,Existing loan takeover,New credit,Application_652823628,20000.0,20000.0,44.0,True,498.29,True,979.0,20000.0,NaN


In [9]:
df.head(10)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,01.01.2016 09:52:36,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Obtained,User_17,W_Complete application,Workflow,Workitem_1875340971,start,2016-01-02 10:45:22.429000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Released,User_17,W_Complete application,Workflow,Workitem_1452291795,suspend,2016-01-02 10:49:28.816000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,02.01.2016 11:23:04,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Created,User_52,O_Create Offer,Offer,Offer_148581083,complete,2016-01-02T11:29:03.994000Z,Existing loan takeover,New credit,Application_652823628,20000.0,20000.0,44.0,True,498.29,True,979.0,20000.0,NaN


In [8]:
# Check how many different formats the df contains now

s = df["time:timestamp"]

patterns = {
    "ISO format YYYY-MM-DD": r"^\d{4}-\d{2}-\d{2}",
    "European slash DD/MM/YYYY": r"^\d{2}/\d{2}/\d{4}",
    "Dash DD-MM-YYYY or MM-DD-YYYY": r"^\d{2}-\d{2}-\d{4}",
    "US-like MM/DD/YYYY": r"^\d{2}/\d{2}/\d{4}",
    "Invalid / custom strings": r"not_a_timestamp|9999|NaT|None"
}

for name, pattern in patterns.items():
    count = s.str.match(pattern, na=False).sum()
    print(name, count)

ISO format YYYY-MM-DD 962480
European slash DD/MM/YYYY 0
Dash DD-MM-YYYY or MM-DD-YYYY 0
US-like MM/DD/YYYY 0
Invalid / custom strings 0


In [11]:
s = df["time:timestamp"].astype(str).str.strip()

iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}", na=False)
eu_mask = s.str.match(r"^\d{1,2}\.\d{1,2}\.\d{4}", na=False)
us_mask = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}", na=False)
utc_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$", na=False)

print("ISO:", iso_mask.sum())
print("European:", eu_mask.sum())
print("US-like:", us_mask.sum())
print("UTC:", utc_mask.sum())
print("unknown:", (~(iso_mask | eu_mask | us_mask)).sum())

ISO: 962480
European: 239373
US-like: 0
UTC: 193657
unknown: 414
